# NB07 — Signal/Pulse: Executive Synthesis
**Signal/Pulse · Portfolio Capstone · Public**

This notebook serves two purposes:
1. **Narrative synthesis** — tells the complete Signal/Pulse story with all four findings in sequence
2. **Asset generator** — pre-computes CSVs and PNGs consumed by `streamlit_app.py`

All charts read from `../outputs/` where possible (produced by NB04–NB06).
Live DB queries are used only for headline numbers and data not already exported.

**Run order:** NB04 → NB05 → NB06 → NB07 → streamlit_app.py

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.image import imread
from IPython.display import display, Image
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from src.schema import get_connection, DB_PATH
from src.viz import apply_style, tight

apply_style()

# ── Palette (mirrors streamlit_app.py) ───────────────────────────────────
C_SKIN    = '#4A90B8'   # calm blue
C_COSM    = '#C4627A'   # dusty rose
C_INGR    = '#5B8C6E'   # sage green
C_KOREAN  = '#D4785C'   # terracotta
C_GOLD    = '#B8965A'   # annotation gold
C_NEUTRAL = '#78909C'

# ── Paths ─────────────────────────────────────────────────────────────────
OUTPUTS = Path('../outputs')
DATA    = Path('../data')
OUTPUTS.mkdir(exist_ok=True)

# ── DB connection ─────────────────────────────────────────────────────────
try:
    conn.execute('SELECT 1')
except Exception:
    conn = get_connection()

# ── Tier helpers (CANON pattern) ──────────────────────────────────────────
CANON      = "COALESCE(p.tier_predicted, p.tier_override, c.tier)"
SKIN_TIERS = ("'skincare','mass_skincare','sensitive_skincare',"
              "'prestige_skincare','sun_protection','dermo_skincare'")
COSM_TIERS = "'cosmetics','mass_cosmetics','prestige_cosmetics','base_makeup'"
ALL_TIERS  = SKIN_TIERS + ',' + COSM_TIERS

# ── Asset check ───────────────────────────────────────────────────────────
REQUIRED_INPUTS = [
    'nb05_slope_chart.png',
    'nb05_trends_shift.png',
    'nb06_cosine_similarity.png',
    'nb06_emerging_terms.png',
    'nb06_blockc_treemap.png',
    'nb06_umap.png',
]

GENERATED_OUTPUTS = [
    # Section 1
    'nb07_trends_crossover.csv',
    'nb07_ingredient_surge.csv',
    'nb07_sku_treemap.csv',
    'nb07_review_slope.csv',
    # Section 2
    'wordcloud_2019.png',
    'wordcloud_2020.png',
    'wordcloud_2021.png',
    'wordcloud_2022.png',
    'wordcloud_2023.png',
    'wordcloud_2024.png',
    'wordcloud_2025.png',
    # Section 3
    'nb07_blockc.csv',
    # umap_embedding.csv — produced by NB06, not NB07
    # YouTube CSVs — produced by NB06 Section 6, verified in Cell 23
    'nb07_language.png',
]

print('INPUT ASSETS (from NB04–NB06):')
for f in REQUIRED_INPUTS:
    status = '✓' if (OUTPUTS / f).exists() else '✗ MISSING — run upstream notebook'
    print(f'  {status}  {f}')

print()
print('OUTPUTS THIS NOTEBOOK WILL GENERATE:')
for f in GENERATED_OUTPUTS:
    status = 'exists' if (OUTPUTS / f).exists() else 'will generate'
    print(f'  [{status:<13}]  {f}')

print()
print(f'DB: {DB_PATH}')
print(f'Reviews : {conn.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]:,}')
print(f'Products: {conn.execute("SELECT COUNT(*) FROM products WHERE source_id=1").fetchone()[0]:,}')

## CELL 1A

## 1. The Shift

Three independent sources confirm the same structural direction: skincare has
overtaken cosmetics as the dominant consumer priority in Japanese beauty post-COVID.
Sources are deliberately independent — different methodologies, different platforms,
different data owners. Convergence across all three is the argument.


## CELL 1B (code) — Headline numbers

In [ ]:
# ── Headline numbers — live from DB ──────────────────────────────────────

mkt_2019 = conn.execute(f"""
    SELECT ROUND(100.0 * SUM(CASE WHEN {CANON} IN ({SKIN_TIERS})
        THEN 1 ELSE 0 END) / COUNT(*), 1)
    FROM reviews r
    JOIN products p   ON r.product_id  = p.product_id
    JOIN categories c ON p.category_id = c.category_id
    WHERE r.review_year = 2019
      AND {CANON} IN ({ALL_TIERS})
""").fetchone()[0]

mkt_2025 = conn.execute(f"""
    SELECT ROUND(100.0 * SUM(CASE WHEN {CANON} IN ({SKIN_TIERS})
        THEN 1 ELSE 0 END) / COUNT(*), 1)
    FROM reviews r
    JOIN products p   ON r.product_id  = p.product_id
    JOIN categories c ON p.category_id = c.category_id
    WHERE r.review_year = 2025
      AND {CANON} IN ({ALL_TIERS})
""").fetchone()[0]

covid_jump = conn.execute(f"""
    WITH yr AS (
        SELECT r.review_year, COUNT(*) AS n
        FROM reviews r
        JOIN products p   ON r.product_id  = p.product_id
        JOIN categories c ON p.category_id = c.category_id
        WHERE {CANON} IN ({SKIN_TIERS})
          AND r.review_year IN (2019, 2020)
        GROUP BY r.review_year
    )
    SELECT ROUND(100.0 * (MAX(n) - MIN(n)) / MIN(n), 1) FROM yr
""").fetchone()[0]

skin_skus = conn.execute(f"""
    SELECT COUNT(*) FROM products p
    JOIN categories c ON p.category_id = c.category_id
    WHERE p.source_id = 1 AND {CANON} IN ({SKIN_TIERS})
""").fetchone()[0]

cosm_skus = conn.execute(f"""
    SELECT COUNT(*) FROM products p
    JOIN categories c ON p.category_id = c.category_id
    WHERE p.source_id = 1 AND {CANON} IN ({COSM_TIERS})
""").fetchone()[0]

sku_ratio = round(skin_skus / cosm_skus, 1)

crossover = conn.execute("""
    WITH annual AS (
        SELECT week_year,
               AVG(CASE WHEN term = 'スキンケア' THEN interest END) AS skin,
               AVG(CASE WHEN term = '化粧品'    THEN interest END) AS cosm
        FROM trends_weekly
        WHERE term_group = 'block_A'
          AND term IN ('スキンケア', '化粧品')
        GROUP BY week_year
    )
    SELECT MIN(week_year) FROM annual WHERE skin >= cosm
""").fetchone()[0]

print('THE SHIFT — HEADLINE NUMBERS')
print('=' * 50)
print(f'  Skincare review share 2019 : {mkt_2019}%')
print(f'  Skincare review share 2025 : {mkt_2025}%')
print(f'  COVID inflection YoY       : +{covid_jump}%')
print(f'  Rakuten SKU ratio          : {sku_ratio}x')
print(f'  Google Trends crossover    : {crossover}')
print(f'  Skincare SKUs              : {skin_skus:,}')
print(f'  Cosmetics SKUs             : {cosm_skus:,}')

## CELL 1C (code) — Export: Google Trends crossover data

In [ ]:
# ── nb07_trends_crossover.csv ─────────────────────────────────────────────
df_crossover = pd.read_sql("""
    SELECT week_start, term, interest
    FROM trends_weekly
    WHERE term_group = 'block_A'
      AND term IN ('スキンケア', '化粧品')
    ORDER BY week_start, term
""", conn)

df_crossover['week_start'] = pd.to_datetime(df_crossover['week_start'])

out = OUTPUTS / 'nb07_trends_crossover.csv'
df_crossover.to_csv(out, index=False, encoding='utf-8-sig')
print(f'Saved → {out.name}')
print(f'  Rows: {len(df_crossover):,}')
print(f'  Date range: {df_crossover.week_start.min().date()} → '
      f'{df_crossover.week_start.max().date()}')
print(f'  Terms: {df_crossover.term.unique().tolist()}')

## CELL 1D (code) — Export: Ingredient surge data

In [ ]:
# ── nb07_ingredient_surge.csv ────────────────────────────────────────────
# Ingredient terms from keywords.xlsx — block_A independent calls only
# block_B excluded: anchored methodology, not suitable for time-series comparison

INGREDIENT_TERMS = [
    'セラミド',
    'ナイアシンアミド',
    'レチノール',
    'ヒアルロン酸',
    'アゼライン酸',
    'エクソソーム',
    'レチナール',
    'グルタチオン',
    'トラネキサム酸',
    'ビタミンC 美容',
]

placeholders = ','.join([f"'{t}'" for t in INGREDIENT_TERMS])

df_ingredients = pd.read_sql(f"""
    SELECT week_start, term, interest
    FROM trends_weekly
    WHERE term_group = 'block_A'
      AND term IN ({placeholders})
    ORDER BY week_start, term
""", conn)

df_ingredients['week_start'] = pd.to_datetime(df_ingredients['week_start'])

out = OUTPUTS / 'nb07_ingredient_surge.csv'
df_ingredients.to_csv(out, index=False, encoding='utf-8-sig')
print(f'Saved → {out.name}')
print(f'  Rows: {len(df_ingredients):,}')
print(f'  Terms ({df_ingredients.term.nunique()}): '
      f'{sorted(df_ingredients.term.unique().tolist())}')
print(f'  Date range: {df_ingredients.week_start.min().date()} → '
      f'{df_ingredients.week_start.max().date()}')

# Sanity check — pre vs post COVID avg per term
print()
print('  Pre-COVID avg (≤2019) vs Post-COVID avg (≥2021):')
df_ingredients['year'] = df_ingredients['week_start'].dt.year
summary = df_ingredients.groupby('term').apply(lambda d: pd.Series({
    'pre_covid':  d[d.year <= 2019]['interest'].mean().round(1),
    'post_covid': d[d.year >= 2021]['interest'].mean().round(1),
})).reset_index()
summary['delta'] = (summary['post_covid'] - summary['pre_covid']).round(1)
summary = summary.sort_values('delta', ascending=False)
print(summary.to_string(index=False))

## CELL 1E (code) — Export: Rakuten SKU treemap data

In [ ]:
# ── nb07_sku_treemap.csv (expanded) ──────────────────────────────────────
df_treemap = pd.read_sql(f"""
    SELECT
        CASE WHEN {CANON} IN ({SKIN_TIERS})
             THEN 'skincare' ELSE 'cosmetics' END AS tier_group,
        c.normalized_name     AS category,
        COUNT(*)              AS sku_count,
        ROUND(AVG(p.review_count), 1)  AS avg_reviews,
        ROUND(AVG(p.review_avg), 2)    AS avg_rating,
        ROUND(AVG(p.price_jpy), 0)     AS avg_price,
        ROUND(MIN(p.price_jpy), 0)     AS min_price,
        ROUND(MAX(p.price_jpy), 0)     AS max_price
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
    WHERE p.source_id = 1
      AND {CANON} IN ({ALL_TIERS})
      AND c.normalized_name != 'beauty_all'  -- beauty_all excluded by design:
      -- NB02b sets tier_predicted on these products, but c.normalized_name
      -- remains 'beauty_all'. Showing catalog by actual category name only.
    GROUP BY tier_group, category
    ORDER BY tier_group, sku_count DESC
""", conn)

out = OUTPUTS / 'nb07_sku_treemap.csv'
df_treemap.to_csv(out, index=False, encoding='utf-8-sig')
print(f'Saved → {out.name}')
print()
print(df_treemap.to_string(index=False))

## CELL 1F (code) — Export: Review slope data

In [ ]:
# ── nb07_review_slope.csv ────────────────────────────────────────────────
df_slope = pd.read_sql(f"""
    WITH yearly_tier AS (
        SELECT
            r.review_year,
            CASE WHEN {CANON} IN ({SKIN_TIERS})
                 THEN 'skincare' ELSE 'cosmetics' END AS tier_group,
            COUNT(*) AS review_count
        FROM reviews r
        JOIN products p   ON r.product_id  = p.product_id
        JOIN categories c ON p.category_id = c.category_id
        WHERE r.review_year BETWEEN 2019 AND 2025
          AND {CANON} IN ({ALL_TIERS})
          AND c.normalized_name != 'beauty_all'  -- see Cell 12 note
        GROUP BY r.review_year, tier_group
    ),
    total_yearly AS (
        SELECT review_year, SUM(review_count) AS total
        FROM yearly_tier
        GROUP BY review_year
    )
    SELECT
        yt.review_year,
        yt.tier_group,
        yt.review_count,
        ROUND(100.0 * yt.review_count / ty.total, 1) AS share_pct
    FROM yearly_tier yt
    JOIN total_yearly ty ON yt.review_year = ty.review_year
    ORDER BY yt.review_year, yt.tier_group
""", conn)

out = OUTPUTS / 'nb07_review_slope.csv'
df_slope.to_csv(out, index=False, encoding='utf-8-sig')
print(f'Saved → {out.name}')
print()
print(df_slope.to_string(index=False))

### Finding 1 — The structural shift is confirmed

> Skincare review share: **85.7% (2019) → 87.3% (2025)**
> COVID inflection: **+152% YoY skincare volume 2019→2020**
> Rakuten SKU ratio: **4.0x** (24,721 skincare vs 6,179 cosmetics)
> Google Trends crossover: **2020** (スキンケア annual avg overtakes 化粧品)
>
> The 2022 cosmetics rebound (+20pp share) was temporary — by 2025
> cosmetics had retreated to 12.7%, the lowest point in the dataset.
> Three independent sources, same direction.
>
> *SKU ratio updated post-NB02b tier classifier. Crossover = first year
> where スキンケア annual avg ≥ 化粧品 annual avg in Block A trends data.
> Weekly chart shows continued divergence through 2023–2025.*


## CELL 2A (FINAL) — TF-IDF weighted word clouds, three variants

In [ ]:
from wordcloud import WordCloud
from sudachipy import tokenizer as tokenizer_obj
from sudachipy import dictionary
from collections import Counter
import re

FONT_PATH = r'C:\Windows\Fonts\msgothic.ttc'
_tokenizer = dictionary.Dictionary().create()
_mode = tokenizer_obj.Tokenizer.SplitMode.C

# ── Brand names from brands.xlsx — suppress in word clouds ───────────────
BRAND_STOPWORDS = {
    'スック', 'アールエムケー', 'センサイ', 'カネボウ', 'エスト',
    'ソフィーナ', 'ケイト', 'フリープラス', 'キュレル', 'ルナソル',
    'アリィー', 'プリマヴィスタ', 'suisai', 'デュウ', 'エビータ',
    'ビオレ', 'ラロッシュポゼ', 'ヴィシー', 'メイベリン', 'ランコム',
    'シュウウエムラ', 'シュウ', 'ウエムラ', '資生堂', 'アネッサ',
    'クレドポー', 'ナーズ', 'イプサ', 'エリクシール', 'マキアージュ',
    'コーセー', 'デコルテ', '雪肌精', 'ヴィセ', '肌ラボ', 'メラノ',
    'セザンヌ', 'キャンメイク', 'エスティローダー', 'クリニーク',
    'ラメール', 'ボビイブラウン', 'トムフォード', 'ディオール',
    'ジバンシィ', 'ゲラン', 'ベネフィット', 'フレッシュ',
    'フェンティ', 'タカミ', 'キールズ', 'オルビス', 'アテニア',
    'ヒロインメイク', 'ヒロイン', 'ドクターケイ',
    # Korean brands
    'アヌア', 'コスアールエックス', 'メディキューブ', 'イニスフリー',
    'ラネージュ', 'クリオ', 'ミシャ',
}

# ── Generic stopwords ─────────────────────────────────────────────────────
STOPWORDS = {
    'する', 'いる', 'ある', 'なる', 'くる', 'いく', 'もの', 'こと',
    'ない', 'れる', 'られる', 'てる', 'です', 'ます', 'でし', 'まし',
    'これ', 'それ', 'あれ', 'この', 'その', 'あの',
    'とても', 'すごく', 'かなり', 'ちょっと', 'また', 'もう',
    'よく', 'さん', 'ちゃん', 'くん', 'におい', 'かな',
    'さらに', 'ほど', 'だけ', 'まま', 'ため', 'よう', 'くらい',
    '良い', 'いい', 'よい', 'すごい', 'ずごい',
    'リピート', 'リピ',
    '購入', '使用', '使い', '使う',
    '感じ', '感じる',
    '商品', 'コスメ', 'プレゼント',
    'ところ', 'アット', 'ほう',
    'こちら', 'そちら', 'あちら',
    'なんか', 'なんと', 'やっぱ', 'やはり',
    'ずっと', 'しっかり', 'しばらく',
    'ない', 'なく', 'なかっ',
    '自分', '今回', '今', '前', '本当',
    'あり', 'あっ', 'なり', 'いつも',
    'thought', 'use', 'the', 'and', 'for', 'fas',
}

ALL_STOPWORDS = STOPWORDS | BRAND_STOPWORDS

# Regex patterns to filter noise tokens
NOISE_PATTERNS = [
    r'^\d+$',               # pure numbers: 00, 01, 101
    r'^\d+[\.\d]*mm$',      # measurements: 0.1mm, 1.5mm
    r'^[\d\.]+$',           # decimals: 0.1, 4.5
    r'^[a-zA-Z]{1,2}$',     # single/double latin chars: mm, cc
    r'^\d+[a-zA-Z]+$',      # mixed: 2020, 3g
]

def is_noise(token):
    for pattern in NOISE_PATTERNS:
        if re.match(pattern, token, re.IGNORECASE):
            return True
    return False

def tokenize_text(text):
    if not text or not isinstance(text, str):
        return []
    morphemes = _tokenizer.tokenize(text, _mode)
    tokens = []
    for m in morphemes:
        pos = m.part_of_speech()[0]
        if pos not in ('名詞', '形容詞'):
            continue
        word = m.dictionary_form()
        if len(word) < 2:
            continue
        if word in ALL_STOPWORDS:
            continue
        if is_noise(word):
            continue
        tokens.append(word)
    return tokens

def make_wordcloud(freq, year, save_path, colormap='RdPu', title_suffix=''):
    # Require minimum frequency of 3 — kills one-off noise
    freq = {w: c for w, c in freq.items() if c >= 3}
    if len(freq) < 10:
        print(f'  WARNING: only {len(freq)} tokens after filtering for {year}')
        return

    wc = WordCloud(
        font_path=FONT_PATH,
        width=1200,
        height=600,
        background_color='white',
        max_words=40,           # fewer = each word larger = more readable
        colormap=colormap,
        prefer_horizontal=0.85,
        min_font_size=12,
        max_font_size=160,
        collocations=False,
    ).generate_from_frequencies(freq)

    fig, ax = plt.subplots(figsize=(12, 6), facecolor='white')
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{year}{title_suffix}',
                 fontsize=18, fontweight='bold',
                 color='#2D2D2D', pad=12)
    tight(fig, str(save_path), dpi=150)
    plt.show()
    top5 = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f'  {save_path.name}  top 5: {[w for w,_ in top5]}')

# ── Load and tokenize ─────────────────────────────────────────────────────
print('Loading reviews...')
df_reviews = pd.read_sql(f"""
    SELECT r.review_year, r.review_text,
           CASE WHEN {CANON} IN ({SKIN_TIERS})
                THEN 'skincare' ELSE 'cosmetics' END AS tier_group
    FROM reviews r
    JOIN products p   ON r.product_id  = p.product_id
    JOIN categories c ON p.category_id = c.category_id
    WHERE r.review_year BETWEEN 2019 AND 2025
      AND r.review_text IS NOT NULL
      AND c.normalized_name != 'beauty_all'
      AND {CANON} IN ({ALL_TIERS})
""", conn)
print(f'Reviews: {len(df_reviews):,}')

print('Tokenizing...')
df_reviews['tokens'] = df_reviews['review_text'].apply(tokenize_text)
print('Done.')

YEARS = list(range(2019, 2026))

# ── Generate: PUBLIC combined ─────────────────────────────────────────────
print('\n--- PUBLIC: Combined word clouds ---')
for year in YEARS:
    yr_tokens = [t for tokens in
                 df_reviews[df_reviews.review_year == year]['tokens']
                 for t in tokens]
    freq = Counter(yr_tokens)
    make_wordcloud(freq, year,
                   OUTPUTS / f'wordcloud_{year}.png',
                   colormap='YlGnBu')

# ── Generate: PRIVATE skincare ────────────────────────────────────────────
# NOTE: 2019–2023 skincare-only word clouds include reviews from
# LDA Skin T3 (mascara miscategorised as skincare) and Skin T4
# (@cosme giveaway templates). マスカラ/細い/まつ毛 appearing in
# skincare clouds pre-2024 is documented corpus noise, not a signal.
print('\n--- PRIVATE: Skincare-only ---')
for year in YEARS:
    yr_tokens = [t for tokens in
                 df_reviews[(df_reviews.review_year == year) &
                             (df_reviews.tier_group == 'skincare')]['tokens']
                 for t in tokens]
    freq = Counter(yr_tokens)
    make_wordcloud(freq, year,
                   OUTPUTS / f'wordcloud_skincare_{year}.png',
                   colormap='Blues',
                   title_suffix=' — skincare')

# ── Generate: PRIVATE cosmetics ───────────────────────────────────────────
print('\n--- PRIVATE: Cosmetics-only ---')
for year in YEARS:
    yr_tokens = [t for tokens in
                 df_reviews[(df_reviews.review_year == year) &
                             (df_reviews.tier_group == 'cosmetics')]['tokens']
                 for t in tokens]
    freq = Counter(yr_tokens)
    make_wordcloud(freq, year,
                   OUTPUTS / f'wordcloud_cosmetics_{year}.png',
                   colormap='RdPu',
                   title_suffix=' — cosmetics')

print('\n' + '=' * 55)
print('Complete. 21 word clouds generated.')

In [ ]:
# ── Auto-copy word clouds to dashboard/assets/ ────────────────────────────
import shutil
DASHBOARD_ASSETS = Path('../dashboard/assets')
print('\nCopying word clouds to dashboard/assets/...')
for year in YEARS:
    src = OUTPUTS / f'wordcloud_{year}.png'
    if src.exists():
        shutil.copy2(src, DASHBOARD_ASSETS / f'wordcloud_{year}.png')
        print(f'  ✓  wordcloud_{year}.png')
print('Done.')

## CELL 2B (markdown)

### Finding 2 — The language changed

The word clouds show the vocabulary shift directly from consumer voice.
2019–2023: makeup application vocabulary dominates — マスカラ, アイライナー, 
まつ毛, メイク. 2024–2025: functional skincare vocabulary takes over — 乾燥, 
保湿, クリーム, 香り.

The cosine similarity and TF-IDF analysis (NB06) quantifies what the word 
clouds show visually: skincare ↔ cosmetics vocabulary overlap rose from 
0.38 (2019) to 0.81 (2023–25).

## CELL 2C (code) — Display NB06 language assets

In [ ]:
# ── Display cosine similarity + emerging terms from NB06 ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='white')

for ax, fname, title in [
    (axes[0], 'nb06_emerging_terms.png',
     'Vocabulary shift — TF-IDF delta 2019→2025\n'
     'Rising: functional/sensory  ·  Declining: visual/makeup'),
    (axes[1], 'nb06_cosine_similarity.png',
     'Vocabulary convergence — cosine similarity 2019→2025\n'
     'Skincare ↔ cosmetics: 0.38 (2019) → 0.81 (2023–25)'),
]:
    p = OUTPUTS / fname
    if p.exists():
        ax.imshow(imread(str(p)))
        ax.axis('off')
        ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    else:
        ax.text(0.5, 0.5, f'{fname}\nnot found',
                ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

fig.suptitle('The language of Japanese beauty — what changed and how much',
             fontsize=13, fontweight='bold')
plt.tight_layout()
tight(fig, str(OUTPUTS / 'nb07_language.png'))
plt.show()

In [ ]:
# ── Verify YouTube assets — produced by NB06 Section 6 Cell 33 ───────────
# NB06 owns YouTube export logic (correct category mapping, 2019–2026 range).
# NB06 Cell 38 copies them to dashboard/assets/.
# This cell just confirms the files are present and non-empty.

import shutil
from pathlib import Path as _P

DASHBOARD_ASSETS = _P('../dashboard/assets')
OUTPUTS_DIR = OUTPUTS

yt_assets = [
    'nb07_yt_volume.csv',
    'nb07_yt_channels.csv',
    'nb07_yt_tfidf.csv',
]

print("YouTube CSV verification:")
all_ok = True
for fname in yt_assets:
    asset_path = DASHBOARD_ASSETS / fname
    out_path   = OUTPUTS_DIR / fname
    if asset_path.exists():
        rows = sum(1 for _ in open(asset_path, encoding='utf-8-sig')) - 1
        # Mirror to outputs/ for consistency
        shutil.copy2(asset_path, out_path)
        print(f"  ✓  {fname}  ({rows} data rows)")
    else:
        print(f"  ✗  {fname}  — MISSING. Re-run NB06 Section 6.")
        all_ok = False

print()
if all_ok:
    print("All YouTube assets present. Dashboard ready.")
else:
    print("Re-run NB06 Sections 6 and 8 (auto-copy) to generate missing files.")


In [ ]:
conn.close()
print()
print('=' * 60)
print('NB07 — Signal/Pulse: Executive Synthesis: COMPLETE')
print('=' * 60)
print()
print('GENERATED OUTPUTS:')
all_out = [
    # Section 1
    'nb07_trends_crossover.csv',
    'nb07_ingredient_surge.csv',
    'nb07_sku_treemap.csv',
    'nb07_review_slope.csv',
    # Section 2
    'wordcloud_2019.png', 'wordcloud_2020.png', 'wordcloud_2021.png',
    'wordcloud_2022.png', 'wordcloud_2023.png', 'wordcloud_2024.png',
    'wordcloud_2025.png',
    'nb07_language.png',
    # YouTube (verified from NB06)
    'nb07_yt_volume.csv', 'nb07_yt_channels.csv', 'nb07_yt_tfidf.csv',
]
for f in all_out:
    p = OUTPUTS / f
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  [{status}]  {f}')
print()
print('Next: streamlit run dashboard/streamlit_app.py')
